# Draw Keyjoint

## PyTorch

In [1]:
import torch
import csv
import json
import os
import pandas as pd
import numpy as np
import torchvision
from torchvision.utils import draw_keypoints, make_grid, draw_bounding_boxes
from PIL import Image
import torchvision.transforms.functional as F
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = torchvision.models.detection.keypointrcnn_resnet50_fpn(pretrained=True)
model.to(device)
model.eval()

c:\Users\denni\OneDrive\Documents\Keypoint_R-CNN\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\denni\OneDrive\Documents\Keypoint_R-CNN\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=KeypointRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=KeypointRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


KeypointRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(640, 672, 704, 736, 768, 800), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.

In [3]:
# refer online
COCO_SKELETON_0_INDEXED = [
    (1, 0), (2, 0),
    (3, 1), (4, 2),
    (5, 6),
    (5, 7), (6, 8),
    (7, 9), (8, 10),
    (5, 11), (6, 12),
    (11, 13), (12, 14),
    (13, 15), (14, 16)
]
parts = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]
column_names = []
for part in parts:
    column_names.extend([f"{part}_x", f"{part}_y", f"{part}_v"])
keypoint_data = []

In [4]:
data_path = Path('Sitting Posture v2.v4i.coco/train/_annotations.coco.json')

with open(data_path, 'r') as f:
    coco_data = json.load(f)

In [5]:
def get_keypoints(image_id):
  #COCO JSON
  img_info = next(item for item in coco_data['images'] if item['id'] == image_id)
  file_name = img_info['file_name']

  ann_info = [item for item in coco_data['annotations'] if item['image_id'] == image_id]

  image_path = os.path.join('Sitting Posture v2.v4i.coco/train/', file_name)

  img = Image.open(image_path).convert("RGB")
  img_tensor = torch.as_tensor(np.array(img)).permute(2, 0, 1)

  bbox = []
  cat_id = 0
  for ann in ann_info:
    x,y,w,h = ann['bbox']
    cat_id = ann['category_id']
    bbox.append([x,y,x+w,y+h])

  bbox_tensor = torch.tensor(bbox, dtype=torch.float32)
  img_with_boxes = draw_bounding_boxes(img_tensor, bbox_tensor, colors='red',width=2)
  img_for_drawing = F.to_tensor(img).unsqueeze(0).to(device)

  with torch.no_grad():
      output = model(img_for_drawing)[0]

  keypoints = output['keypoints']
  keypoints_scores = output['keypoints_scores']

  #Skip if empty result
  if (keypoints.shape[0] == 0) : return
  scores = output['scores']

  # 90% con7firm result
  high_score_kp = keypoints[scores > 0.9]
  high_score_kp_scores = keypoints_scores[scores > 0.9]
  #debug
  print(scores)
  print(high_score_kp)

  if high_score_kp.shape[0] == 0: return

  #if more than one
  selected_kp = high_score_kp[0]
  selected_kp_scores = high_score_kp_scores[0]
  print(selected_kp_scores)

  kp_filtered = selected_kp.clone()

  for i in range(17):
    if selected_kp_scores[i] < 0.0:
        kp_filtered[i, :] = torch.tensor([0.0, 0.0, 0.0], device=kp_filtered.device)

  #store csv
  data_row = kp_filtered[:, :3].reshape(-1).cpu().numpy()
  df = pd.DataFrame([data_row], columns=column_names)
  df.insert(0, 'image_id', image_id)
  df['cat_id'] = cat_id
  keypoint_data.append(df)

  csv_filename = "keypoint_train_data.csv"
  file_exists = os.path.isfile(csv_filename)

  # Combine the data into a single standard Python list
  full_row = [image_id, cat_id] + data_row.tolist()

  # Open the file and write directly
  with open(csv_filename, mode='a', newline='') as f:
    writer = csv.writer(f)
    
    # Write the header only if the file was just created
    if not file_exists:
      writer.writerow(['image_id', 'cat_id'] + column_names) 
        
    writer.writerow(full_row)
    
  #draw kp on image

  if len(selected_kp[0]) > 0:
    pred_keypoints, visibility = kp_filtered.split([2, 1], dim=-1)
    visibility = visibility.bool()

    annotated_img = draw_keypoints(
        img_with_boxes,
        pred_keypoints.unsqueeze(0),
        visibility=visibility.unsqueeze(0),
        colors='red',
        radius=4,
        connectivity=COCO_SKELETON_0_INDEXED
    )

  #just tengok
  grid = make_grid(annotated_img)
  grid_np = grid.permute(1, 2, 0).numpy() #(H,W,C)

  plt.figure(figsize=(10, 10))
  plt.imshow(grid_np)
  plt.title("Torch!")
  plt.axis('off')
  plt.show()

  output_filename = "keyjoint_train_dataset/keyjoints_" + os.path.basename(image_path)
  torchvision.io.write_png((annotated_img).to(torch.uint8), output_filename)

In [6]:
import os

# Create the output directory if it doesn't exist
output_dir = "keyjoint_train_dataset"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

for image_info in coco_data['images']:
    image_id = image_info['id']
    print(f"Processing image ID: {image_id}")
    get_keypoints(image_id=image_id)

Processing image ID: 0


KeyboardInterrupt: 

In [ ]:
get_keypoints(image_id=2)

In [ ]:
combined_keypoint_df = pd.concat(keypoint_data, ignore_index=True)
combined_keypoint_df.to_csv("keypoint_train_data.csv", index=False)

In [ ]:
!zip -r /content/keyjoint_train_dataset.zip /content/keyjoint_train_dataset

# 1.0 Import Data

In [ ]:
# %%capture
!pip -q install torchinfo

In [ ]:
# Import Library
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import json
import warnings

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.utils.data import SubsetRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.transforms import functional as F
from torch.amp import GradScaler, autocast
from torchinfo import summary

# Scikit-learn imports
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Additional utilities
import cv2
from tqdm import tqdm
import gc
from collections import Counter

# Check GPU availability
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

In [ ]:
with open('/content/_annotations.coco.json', 'r') as f:
    coco_data = json.load(f)

In [ ]:
!unzip "/content/sitting-posture-keyjoint-dataset.zip" -d "/content/sitting-posture-keyjoint-dataset"

In [ ]:
train_df = pd.read_csv('/content/sitting-posture-keyjoint-dataset/keyjoint_train_dataset/keypoint_train_data.csv')
valid_df = pd.read_csv('/content/sitting-posture-keyjoint-dataset/keyjoint_valid_dataset/keypoint_valid_data.csv')
test_df = pd.read_csv('/content/sitting-posture-keyjoint-dataset/keyjoint_test_dataset/keypoint_test_data.csv')

# 2.0 EDA and Data Preprocessing

#### Exploratory and Data Analysis

##### Data Description

In [ ]:
# Training data information
print("=== TRAIN DATA INFO ===")
print(f"Train data shape: {train_df.shape}")
print(f"Columns: {train_df.columns}")
print(f"Data types: \n{train_df.dtypes}")
print(f"-1 values: \n{(train_df == -1).sum()}")

# Valid data information
print("\n=== TEST DATA INFO ===")
print(f"Train data shape: {valid_df.shape}")
print(f"Columns: {valid_df.columns}")
print(f"-1 values: \n{(valid_df == -1).sum()}")

# Testing data information
print("=== TEST DATA INFO ===")
print(f"Train data shape: {test_df.shape}")
print(f"Columns: {test_df.columns}")
print(f"-1 values: \n{(test_df == -1).sum()}")

In [ ]:
#Split X_train, y_train
X_train = train_df.drop(['cat_id'], axis=1)
y_train = train_df['cat_id']
X_valid = valid_df.drop(['cat_id'], axis=1)
y_valid = valid_df['cat_id']
X_test = test_df.drop(['cat_id'], axis=1)
y_test = test_df['cat_id']

In [ ]:
y_train.value_counts()

##### Data Visualization (For valid keypoint)

In [ ]:
missing_df = (X_train == -1)

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(missing_df, cbar=False, cmap='viridis')
plt.title("Missing Keypoints Pattern (Yellow = Missing(-1))")
plt.show()

df = train_df.copy()

df['missing_count'] = (df == -1).sum(axis=1)
sns.boxplot(data=df, x='cat_id', y='missing_count')
plt.title("Missing Points Count: Good (2) vs Bad (1)")
plt.show()

1) Since the missing value has been replaced with (-1,-1), check if there are too many missing keypoint in one data

In [ ]:
value_count_df = train_df.copy()
X_columns = [col for col in value_count_df.columns if '_x' in col]
value_count_df['valid_points_count'] = (value_count_df[X_columns] != -1).sum(axis=1)

print(value_count_df['valid_points_count'].value_counts())

From above result, can see that there are some images have less than 12 keypoints.

In [ ]:
filtered_df = value_count_df[value_count_df['valid_points_count'] <= 11]

print("Images with 'valid_points_count' <= 10 and their COCO data:\n")

for index, row in filtered_df.iterrows():
    image_id = int(row['image_id'])
    valid_points_count = row['valid_points_count']

    # Find the corresponding image info in coco_data
    image_info = next((item for item in coco_data['images'] if item['id'] == image_id), None)

    if image_info:
        file_name = image_info['file_name']
        # Correct the image_path to include 'keyjoints_' as part of the filename
        image_path = os.path.join('/content/sitting-posture-keyjoint-dataset/keyjoint_train_dataset', 'keyjoints_' + file_name)
        print(f"Image ID: {image_id}")
        print(f"  File Name: {image_path}")
        print(f"  Valid Points Count: {valid_points_count:.2f}")
        try:
            img = Image.open(image_path)
            plt.figure(figsize=(8, 8)) # Create a new figure for each image
            plt.imshow(img)
            plt.title(f"Image ID: {image_id} - Valid Points Count: {valid_points_count:.2f}")
            plt.axis('off')
            plt.show()
        except FileNotFoundError:
            print(f"  Error: Annotated image not found at {image_path}")
        # You can add more details from row or image_info if needed
        print("------------------------------------------")
    else:
        print(f"Image ID: {image_id} - No matching image info found in coco_data.")
        print(f"  Valid Points Count: {valid_points_count:.2f}")
        print("------------------------------------------")

After checking, consider dropping images with less than 12 valid keypoints

2) Some joint are important to exist (nose, shoulder, hip)

In [ ]:
missing_core_df = train_df.copy()
core_joints = ['nose_x', 'left_shoulder_x', 'right_shoulder_x', 'left_hip_x', 'right_hip_x']
is_missing_core = (missing_core_df[core_joints] == -1).any(axis=1)
missing_core_df = missing_core_df[~is_missing_core].copy()
print(f"Delete {is_missing_core.sum()} records")

##### Feature Distribution - Nose and shoulder distance

In [ ]:
df_valid = train_df[(train_df['nose_x'] != -1) & (train_df['left_shoulder_x'] != -1) & (train_df['right_shoulder_x'] != -1)].copy()

# Calculate shoulder center point
df_valid['sh_center_x'] = (df_valid['left_shoulder_x'] + df_valid['right_shoulder_x']) / 2

df_valid['head_forward'] = abs(df_valid['nose_x'] - df_valid['sh_center_x'])

plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_valid, x='head_forward', hue='cat_id', fill=True)
plt.title("Head Forward Displacement: Good vs Bad")
plt.xlabel("Distance between Nose and Shoulder Center")
plt.show()

Display the "weird" image

In [ ]:
filtered_df = df_valid[df_valid['head_forward'] > 200]

print("Images with 'head_forward' > 200:\n")

for index, row in filtered_df.iterrows():
    image_id = int(row['image_id'])
    head_forward_val = row['head_forward']

    image_info = next((item for item in coco_data['images'] if item['id'] == image_id), None)

    if image_info:
        file_name = image_info['file_name']
        image_path = os.path.join('/content/sitting-posture-keyjoint-dataset/keyjoint_train_dataset', 'keyjoints_' + file_name)
        print(f"Image ID: {image_id}")
        print(f"File Name: {image_path}")
        print(f"Head Forward: {head_forward_val:.2f}")
        try:
            img = Image.open(image_path)
            plt.figure(figsize=(8, 8))
            plt.imshow(img)
            plt.title(f"Image ID: {image_id} - Head Forward: {head_forward_val:.2f}")
            plt.axis('off')
            plt.show()
        except FileNotFoundError:
            print(f"  Error: Annotated image not found at {image_path}")
        print("------------------------------------------")
    else:
        print(f"Image ID: {image_id} - No matching image info found in coco_data.")
        print(f"Head Forward: {head_forward_val:.2f}")
        print("------------------------------------------")

So: Consider dropping images where head_forward > 200

##### Feature Distribution - Left and Right Shoulder Distance

In [ ]:
df_shoulder_dist = train_df.copy()

# Calculate shoulder distance, handling cases where shoulder keypoints might be missing (-1)
df_shoulder_dist['left_shoulder_x'] = df_shoulder_dist['left_shoulder_x'].replace(-1, np.nan)
df_shoulder_dist['left_shoulder_y'] = df_shoulder_dist['left_shoulder_y'].replace(-1, np.nan)
df_shoulder_dist['right_shoulder_x'] = df_shoulder_dist['right_shoulder_x'].replace(-1, np.nan)
df_shoulder_dist['right_shoulder_y'] = df_shoulder_dist['right_shoulder_y'].replace(-1, np.nan)

# Only calculate if both left and right shoulder points exist
df_shoulder_dist = df_shoulder_dist.dropna(subset=['left_shoulder_x', 'left_shoulder_y', 'right_shoulder_x', 'right_shoulder_y'])

df_shoulder_dist['sh_dist'] = ((df_shoulder_dist['left_shoulder_x'] - df_shoulder_dist['right_shoulder_x'])**2 +
                             (df_shoulder_dist['left_shoulder_y'] - df_shoulder_dist['right_shoulder_y'])**2)**0.5


plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_shoulder_dist, x='sh_dist', hue='cat_id', fill=True, common_norm=False)
plt.title('Distribution of Shoulder Distance by Posture Category (Good vs. Bad)')
plt.xlabel('Shoulder Distance (pixels)')
plt.ylabel('Density')
plt.legend(title='Category', labels=['Bad Posture (1)', 'Good Posture (2)'])
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
def is_unusual(row):
  image_id = int(row.image_id)
  sh_dist = ((row.left_shoulder_x - row.right_shoulder_x)**2 +
              (row.left_shoulder_y - row.right_shoulder_y)**2)**0.5

  if sh_dist > 300:
  #if sh_dist > 200 and sh_dist < 300:
    image_info = next((item for item in coco_data['images'] if item['id'] == image_id), None)
    file_name = image_info['file_name']
    image_path = os.path.join('/content/sitting-posture-keyjoint-dataset/keyjoint_train_dataset', 'keyjoints_' + file_name)
    img = Image.open(image_path)
    plt.figure(figsize=(8, 8)) # Create a new figure for each image
    plt.imshow(img)
    plt.title(f"Image ID: {image_id} - Shoulder distance: {sh_dist:.2f}")
    plt.axis('off')
    plt.show()
    return True

  return False

In [ ]:
# !Change later
for row in train_df.itertuples():
  if is_unusual(row):
    print(row)

All images that shoulder distance > 300 are unusual, consider drop it

In [ ]:
def is_unusual2(row):
  image_id = int(row["image_id"])
  sh_dist = ((row["left_shoulder_x"] - row["right_shoulder_x"])**2 +
              (row['left_shoulder_y'] - row['right_shoulder_y'])**2)**0.5

  if sh_dist < 1:
    image_info = next((item for item in coco_data['images'] if item['id'] == image_id), None)
    file_name = image_info['file_name']
    image_path = os.path.join('/content/sitting-posture-keyjoint-dataset/keyjoint_train_dataset', 'keyjoints_' + file_name)
    img = Image.open(image_path)
    plt.figure(figsize=(8, 8)) # Create a new figure for each image
    plt.imshow(img)
    plt.title(f"Image ID: {image_id} - Shoulder distance: {sh_dist:.2f}")
    plt.axis('off')
    plt.show()
    return True

  return False

In [ ]:
for index, row_data in train_df.iterrows():
  if is_unusual2(row_data):
    print(row_data)

All images that shoulder distance < 1 are unusual, consider drop it

Average Pose Visualization

In [ ]:
def plot_average_pose(df, title):
  """
  """
  temp = df.replace(-1, pd.NA).dropna(axis=0, how='any')
  mean_coord = temp.mean()

  connections = [
      ('nose_x', 'nose_y', 'left_eye_x', 'left_eye_y'),
      ('nose_x', 'nose_y', 'right_eye_x', 'right_eye_y'),
      ('left_ear_x', 'left_ear_y', 'left_eye_x', 'left_eye_y'),
      ('right_ear_x', 'right_ear_y', 'right_eye_x', 'right_eye_y'),
      ('left_shoulder_x', 'left_shoulder_y', 'right_shoulder_x', 'right_shoulder_y'),
      ('left_shoulder_x', 'left_shoulder_y', 'left_elbow_x', 'left_elbow_y'),
      ('right_shoulder_x', 'right_shoulder_y', 'right_elbow_x', 'right_elbow_y'),
      ('left_elbow_x', 'left_elbow_y', 'left_wrist_x', 'left_wrist_y'),
      ('right_elbow_x', 'right_elbow_y', 'right_wrist_x', 'right_wrist_y'),
      ('left_shoulder_x', 'left_shoulder_y', 'left_hip_x', 'left_hip_y'),
      ('right_shoulder_x', 'right_shoulder_y', 'right_hip_x', 'right_hip_y'),
      ('left_hip_x', 'left_hip_y', 'left_knee_x', 'left_knee_y'),
      ('right_hip_x', 'right_hip_y', 'right_knee_x', 'right_knee_y'),
      ('left_knee_x', 'left_knee_y', 'left_ankle_x', 'left_ankle_y'),
      ('right_knee_x', 'right_knee_y', 'right_ankle_x', 'right_ankle_y')]
  for x1, y1, x2, y2 in connections:
    plt.plot([mean_coord[x1], mean_coord[x2]], [mean_coord[y1], mean_coord[y2]], marker='o')
    plt.gca().invert_yaxis()
    plt.title(title)

plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plot_average_pose(train_df[train_df['cat_id'] == 1], "Average Bad(1) Pose Visualization")
plt.subplot(1, 2, 2)
plot_average_pose(train_df[train_df['cat_id'] == 2], "Average Good(2) Pose Visualization")
plt.show()

Joint Scatter Plot - Using nose position

In [ ]:
df_nose = train_df[train_df['nose_x'] != -1]

plt.figure(figsize=(8, 8))
sns.scatterplot(data=df_nose, x='nose_x', y='nose_y', hue='cat_id', alpha=0.5, palette=['green', 'red'])
plt.gca().invert_yaxis()
plt.title("Scatter Cloud of Nose Positions")
plt.show()

#### Data Preprocessing

##### Remove unusual rows

Head and Shoulder distance more than 200

In [ ]:
# Keep head_forward < 200 in df
filtered_df = df_valid[df_valid['head_forward'] < 200]

#Debug
original_train_rows = train_df.shape[0]

print(f"Removed {original_train_rows - filtered_df.shape[0]} rows where 'head_forward' was > 200.")

# Update X_train and y_train after filtering
X_train_filtered = filtered_df.drop(['cat_id'], axis=1)
y_train_filtered = filtered_df['cat_id']

print(f"New X_train shape: {X_train_filtered.shape}")
print(f"New y_train shape: {y_train_filtered.shape}")

Left and Right Shoulder distance more than 300 or less than 1

In [ ]:
def is_unusual_shoulder_dist(row):
  image_id = int(row.image_id)
  if row.left_shoulder_x == -1 or row.right_shoulder_x == -1 or \
     row.left_shoulder_y == -1 or row.right_shoulder_y == -1:
      return False

  sh_dist = ((row.left_shoulder_x - row.right_shoulder_x)**2 +
              (row.left_shoulder_y - row.right_shoulder_y)**2)**0.5

  if sh_dist > 300 or sh_dist < 1:
    return True

  return False

In [ ]:
filtered_df_drop = filtered_df.copy()

original_train_rows_after_head_filter = train_df.shape[0]

not_unusual_shoulder_mask = ~filtered_df_drop.apply(is_unusual_shoulder_dist, axis=1)

filtered_df_drop = filtered_df_drop[not_unusual_shoulder_mask]

#Debug
print(f"Removed {original_train_rows_after_head_filter - filtered_df_drop.shape[0]} rows where shoulder distance was > 300 or < 1.")

X_train = filtered_df_drop.drop(['cat_id'], axis=1)
y_train = filtered_df_drop['cat_id']

print(f"Final X_train shape: {X_train.shape}")
print(f"Final y_train shape: {y_train.shape}")

In [ ]:
filtered_df_drop_12 = filtered_df_drop.copy()
X_columns = [col for col in filtered_df_drop_12.columns if '_x' in col]
filtered_df_drop_12['valid_points_count'] = (filtered_df_drop_12[X_columns] != -1).sum(axis=1)

filtered_df_drop_12 = filtered_df_drop_12[filtered_df_drop_12['valid_points_count'] > 11]

#Debug
print(f"Removed {filtered_df_drop.shape[0] - filtered_df_drop_12.shape[0]} rows where valid keypoints <= 11")

X_train = filtered_df_drop_12.drop(['cat_id'], axis=1)
y_train = filtered_df_drop_12['cat_id']

print(f"Final X_train shape: {X_train.shape}")
print(f"Final y_train shape: {y_train.shape}")

In [ ]:
filtered_df_drop_12_core = filtered_df_drop_12.copy()
core_joints = ['nose_x', 'left_shoulder_x', 'right_shoulder_x', 'left_hip_x', 'right_hip_x']
is_missing_core = (filtered_df_drop_12_core[core_joints] == -1).any(axis=1)
filtered_df_drop_12_core = filtered_df_drop_12_core[~is_missing_core].copy()
#Debug
print(f"Delete {is_missing_core.sum()} records")

X_train = filtered_df_drop_12_core.drop(['cat_id'], axis=1)
y_train = filtered_df_drop_12_core['cat_id']

print(f"Final X_train shape: {X_train.shape}")
print(f"Final y_train shape: {y_train.shape}")

Visualize Data #Debug

In [ ]:
#Debug
def display_image(row):
  image_id = int(row.image_id)
  image_info = next((item for item in coco_data['images'] if item['id'] == image_id), None)
  file_name = image_info['file_name']
  image_path = os.path.join('/content/sitting-posture-keyjoint-dataset/keyjoint_train_dataset', 'keyjoints_' + file_name)
  img = Image.open(image_path)
  plt.figure(figsize=(8, 8)) # Create a new figure for each image
  plt.imshow(img)
  plt.title(f"Image ID: {image_id}")
  plt.axis('off')
  plt.show()

In [ ]:
#Debug
for index, row_data in filtered_df_drop_12_core.iterrows():
  display_image(row_data)

After checking, still got 12 images with unusual pattern, manually delete it

In [ ]:
ususual_id_list = [1227,1221,1175,1083,1010,863,672,626,623,614,560,540]
final_df = filtered_df_drop_12_core[~filtered_df_drop_12_core['image_id'].isin(ususual_id_list)]

In [ ]:
X_train = final_df.drop(['cat_id'], axis=1)
y_train = final_df['cat_id']

print(f"Final X_train shape: {X_train.shape}")
print(f"Final y_train shape: {y_train.shape}")

In [ ]:
#Debug
y_train.value_counts()

##### Normalize 0 ~ 1, for valid keypoint

In [ ]:
features = [col for col in X_train.columns if col != 'image_id']

In [ ]:
# map -1 to nan
temp_train_train = X_train[features].replace(-1, np.nan)
# temp_valid_train = X_valid[features].replace(-1, np.nan)
# temp_test_train = X_test[features].replace(-1, np.nan)

In [ ]:
scaler = MinMaxScaler()
scaler.fit(temp_train_train)

In [ ]:
X_train_norm = X_train.copy()
# X_valid_norm = X_valid.copy()
# X_test_norm = X_test.copy()

In [ ]:
X_train_norm[features] = scaler.transform(temp_train_train)
# X_valid_norm[features] = scaler.transform(temp_valid_train)
# X_test_norm[features] = scaler.transform(temp_test_train)

In [ ]:
X_train_norm = X_train_norm.fillna(-1, inplace=True)
# X_valid_norm = X_valid_norm.fillna(-1, inplace=True)
# X_test_norm = X_test_norm.fillna(-1, inplace=True)

In [ ]:
#...#Debug
X_train_norm = X_train_norm.drop(['image_id'], axis=1)